# 📦 Cenário de Dados: Sistema de Despachos

Este cenário simula o banco de dados de um aplicativo de logística e despachos. Os dados são altamente transacionais (mudanças constantes de status de motoristas e corridas), justificando o uso de tecnologias Lakehouse como o Delta Lake para garantir transações ACID.

## Modelo Entidade-Relacionamento (ER)
Abaixo, o relacionamento entre os motoristas cadastrados e as corridas realizadas.

```mermaid
erDiagram
    MOTORISTAS ||--o{ CORRIDAS : "realiza"
    
    MOTORISTAS {
        int id_motorista PK
        string nome
        string veiculo
        string status
    }
    
    CORRIDAS {
        int id_corrida PK
        int id_motorista FK
        string origem
        string destino
        float valor_corrida
        string status_corrida
    }

In [2]:
import os
import urllib.request
import pyspark
from delta import *

hadoop_home = os.path.abspath("hadoop_win")
hadoop_bin = os.path.join(hadoop_home, "bin")
os.makedirs(hadoop_bin, exist_ok=True)

base_url = "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.2.0/bin/"

for file in ["winutils.exe", "hadoop.dll"]:
    file_path = os.path.join(hadoop_bin, file)
    if not os.path.exists(file_path):
        print(f"Baixando {file} para o Windows parar de reclamar...")
        urllib.request.urlretrieve(base_url + file, file_path)

os.environ["HADOOP_HOME"] = hadoop_home
os.environ["PATH"] += os.pathsep + hadoop_bin
# =====================================================================


print("Ligando o motor do Apache Spark...")
builder = pyspark.sql.SparkSession.builder.appName("DeltaLake_Despachos") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR") # Esconde os avisos vermelhos gigantes

print("Spark com Delta Lake iniciado com sucesso!")

# 2. Comandos DDL: Criando as Tabelas
spark.sql("""
CREATE TABLE IF NOT EXISTS motoristas (
    id_motorista INT,
    nome STRING,
    veiculo STRING,
    status STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS corridas (
    id_corrida INT,
    id_motorista INT,
    origem STRING,
    destino STRING,
    valor_corrida FLOAT,
    status_corrida STRING
) USING DELTA
""")

print("✅ Tabelas 'motoristas' e 'corridas' criadas no formato Delta!")

Ligando o motor do Apache Spark...
:: loading settings :: url = jar:file:/home/bruno/Projeto_Eng_de_dados/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/bruno/.ivy2/cache
The jars for the packages stored in: /home/bruno/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0f132c19-5001-41eb-bdcc-9246dc295d18;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (770ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (84ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (111ms)
:: resolution report :: resolve 3750ms :: ar

Spark com Delta Lake iniciado com sucesso!


AnalysisException: [DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION] Cannot create table ('`default`.`motoristas`'). The associated location ('file:/home/bruno/Projeto_Eng_de_dados/spark-warehouse/motoristas') is not empty and also not a Delta table.

In [5]:
print("Iniciando operações no Delta Lake...\n")

print(" INSERT: Adicionando motoristas e corridas...")
spark.sql("""
INSERT INTO motoristas VALUES 
(1, 'João Silva', 'Fiat Uno', 'Ativo'),
(2, 'Maria Souza', 'Chevrolet Onix', 'Inativo')
""")

spark.sql("""
INSERT INTO corridas VALUES 
(101, 1, 'Centro', 'Aeroporto', 45.50, 'Em Andamento')
""")

print("Tabela Motoristas após o INSERT:")
spark.sql("SELECT * FROM motoristas").show()

print(" UPDATE: Atualizando o status da corrida para 'Finalizada'...")
spark.sql("""
UPDATE corridas 
SET status_corrida = 'Finalizada', valor_corrida = 50.00
WHERE id_corrida = 101
""")

print("Tabela Corridas após o UPDATE:")
spark.sql("SELECT * FROM corridas").show()

print(" DELETE: Removendo motoristas inativos do sistema...")
spark.sql("""
DELETE FROM motoristas 
WHERE status = 'Inativo'
""")

print("Tabela Motoristas após o DELETE:")
spark.sql("SELECT * FROM motoristas").show()

Iniciando operações no Delta Lake...

 INSERT: Adicionando motoristas e corridas...
Tabela Motoristas após o INSERT:
+------------+-----------+--------------+-------+
|id_motorista|       nome|       veiculo| status|
+------------+-----------+--------------+-------+
|           2|Maria Souza|Chevrolet Onix|Inativo|
|           1| João Silva|      Fiat Uno|  Ativo|
|           1| João Silva|      Fiat Uno|  Ativo|
|           1| João Silva|      Fiat Uno|  Ativo|
+------------+-----------+--------------+-------+

 UPDATE: Atualizando o status da corrida para 'Finalizada'...
Tabela Corridas após o UPDATE:
+----------+------------+------+---------+-------------+--------------+
|id_corrida|id_motorista|origem|  destino|valor_corrida|status_corrida|
+----------+------------+------+---------+-------------+--------------+
|       101|           1|Centro|Aeroporto|         50.0|    Finalizada|
|       101|           1|Centro|Aeroporto|         50.0|    Finalizada|
|       101|           1|Cent